# R3_04 — Inferensi data uji (DIJALANKAN SATU KALI)

* **`DRY_RUN = True` (default):** seluruh alur dijalankan dengan **data dev** sebagai pengganti data uji, ke `outputs_round3/dry_run/`.
  Data uji tidak disentuh (penjaga aktif). Pakai ini untuk menguji kode.
* **`DRY_RUN = False`:** mode nyata. Sel pertama memeriksa `frozen_choices.json` (kode, data, checkpoint harus identik dengan saat
  dibekukan) dan berhenti bila tidak cocok atau bila R3_04 sudah pernah dijalankan. Label uji **tidak dimuat** (hanya kolom
  `pair_id, tweet_id, sentence_a, sentence_b`) dan **tidak ada metrik** yang dihitung di notebook ini.

Isi: inferensi 9 checkpoint (fungsi `evaluate` NB03, `MAX_LENGTH` 80, FP32) → agregasi seed (`aggregate_seeds` NB03) → hard vote dan
weighted soft vote dengan bobot dev (sel NB05) → penandaan kasus LLM (`needs_llm_judge` NB06: vote terpecah atau margin < 0,15) →
inferensi ablasi V1–V6 → prediksi skema Tabel 17 terpilih. Waktu: ±5–15 menit di GPU.

In [ ]:
# ── PARAMETER ──
DRY_RUN = True

import os
DRY_RUN = os.environ.get('R3_DRY_RUN', str(DRY_RUN)) == 'True'

In [ ]:
import sys, json, gc
sys.path.insert(0, os.environ.get('R3_CODE_DIR', os.getcwd()))
import numpy as np, pandas as pd
import config as C
import r3_lib as L
D = L.mode_dirs(DRY_RUN)
if DRY_RUN:
    L.forbid_test_access(allow_dirs=[C.DRY_DIR])
    print('MODE DRY_RUN — memakai data DEV sebagai pengganti; data uji tidak disentuh')
else:
    print('MODE NYATA — data uji (tanpa label) akan diinferensi SATU KALI')
frozen = L.verify_frozen(strict=not DRY_RUN)
OUT = C.assert_writable(D.test_pred)
if not DRY_RUN and (OUT / 'DONE.json').exists():
    raise RuntimeError(f'R3_04 sudah pernah dijalankan ({OUT / "DONE.json"}). Inferensi uji hanya satu kali.')
OUT.mkdir(parents=True, exist_ok=True)
L.write_json(OUT / 'ACCESS_LOG.json', {'started_at': L.now_utc(), 'dry_run': DRY_RUN, 'frozen_created_at': frozen['created_at'],
                                       'environment': L.environment_info()})
import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Output:', OUT, '| device:', DEVICE)

In [ ]:
# Data target TANPA kolom label
df_t = pd.read_csv(D.pairs, usecols=['pair_id', 'tweet_id', 'sentence_a', 'sentence_b'], dtype={'tweet_id': str})
df_t['pair_id'] = df_t['pair_id'].astype(str)
for col in ['sentence_a', 'sentence_b']:
    df_t[col] = df_t[col].astype(str)
df_t['aspek'] = df_t['pair_id'].map(lambda p: next(a for a in C.ASPECTS if p.endswith('_' + a)))
assert 'label' not in df_t.columns
texts = pd.read_csv(C.TWEETS_PREPROCESSED, usecols=['tweet_id'] + [f'text_{v}' for v in C.TEXT_VERSIONS_ABLATION], dtype={'tweet_id': str})
dv = df_t.merge(texts, on='tweet_id', how='left')
assert len(dv) == len(df_t) and (dv.text_v1.astype(str) == dv.sentence_a).all()
print(f'{len(df_t)} pasangan, {df_t.tweet_id.nunique()} tweet')

## Inferensi 9 checkpoint (V1 = utama) dan ablasi V1–V6

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
t_ns = L.exec_legacy(['nb03_dataset', 'nb03_train_functions'], L.base_namespace())
crit = torch.nn.CrossEntropyLoss()
(OUT / 'ablation').mkdir(exist_ok=True)
abl = {}
for m in C.MODELS:
    for s in C.SEEDS:
        bd = os.path.dirname(frozen['runs'][f'{m}_seed{s}']['best_model_path'])
        tok = AutoTokenizer.from_pretrained(bd)
        mdl = AutoModelForSequenceClassification.from_pretrained(bd, num_labels=C.NUM_LABELS, ignore_mismatched_sizes=True).to(DEVICE)
        for v in C.TEXT_VERSIONS_ABLATION:
            dfv = dv[['pair_id', 'sentence_b']].assign(sentence_a=dv[f'text_{v}'].fillna('').astype(str))
            loader = torch.utils.data.DataLoader(t_ns['SentencePairDataset'](dfv, tok, C.MAX_LENGTH), batch_size=C.BATCH_SIZE, shuffle=False)
            _, _, preds, probs, ids, labels = t_ns['evaluate'](mdl, loader, crit, DEVICE)
            assert labels == [] and list(ids) == list(dfv.pair_id)        # tanpa label → tanpa metrik
            if v == 'v1':
                t_ns['save_predictions'](ids, probs, preds, labels, str(OUT / f'{m}_seed{s}_test.csv'))
            abl.setdefault((m, v), []).append(np.array(probs))
        print(f'  ✓ {m} seed {s}')
        del mdl, tok; gc.collect()
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
for (m, v), lst in abl.items():
    avg = np.mean(lst, axis=0)
    pd.DataFrame({'pair_id': dv.pair_id, **{c: avg[:, i] for i, c in enumerate(C.PROB_COLS)}, 'pred_label': avg.argmax(1)}).to_csv(
        OUT / 'ablation' / f'{m}_{v}_test_avg.csv', index=False, encoding='utf-8-sig')

## Agregasi seed, ensemble (bobot dev), penandaan kasus LLM

In [ ]:
# Sel asli NB03 juga menjalankan loop agregasi (split dev & test) di PRED_DIR = OUT; tidak ada file dev di sana →
# split dev hanya mencetak 'tidak ditemukan'. Tanpa label → tidak ada macro-F1 (NaN).
agg_ns = L.exec_legacy('nb03_aggregate_seeds', L.base_namespace(PRED_DIR=str(OUT), SEEDS=C.SEEDS))
test_preds = {}
for m in C.MODELS:
    d = agg_ns['aggregate_seeds'](m, 'test', C.SEEDS, str(OUT))
    d.to_csv(OUT / f'{m}_test_avg.csv', index=False, encoding='utf-8-sig')
    d['pair_id'] = d['pair_id'].astype(str)
    test_preds[m] = d
e_ns = L.exec_legacy(['nb05_vectorized_ensemble', 'nb05_run_ensemble', 'nb05_ensemble_frame'],
                     L.base_namespace(test_preds=test_preds, df_test=df_t, model_weights=frozen['model_weights'], DEVICE=DEVICE,
                                      PRED_DIR=str(OUT)))
ens = pd.read_csv(OUT / 'ensemble_test.csv')
ens['pair_id'] = ens['pair_id'].astype(str)
assert (ens.pair_id.values == df_t.pair_id.values).all() and ens.true_label.isna().all()
p_ns = L.exec_legacy('nb06_prompt', L.base_namespace())
flags = ens[['pair_id', 'aspek', 'vote_type', 'soft_vote_margin']].copy()
flags['needs_llm'] = [p_ns['needs_llm_judge'](v, mg, C.MARGIN_THRESHOLD) for v, mg in zip(flags.vote_type, flags.soft_vote_margin)]
flags.to_csv(OUT / 'flags.csv', index=False, encoding='utf-8-sig')
print(f"Vote terpecah: {(flags.vote_type == 'terpecah').sum()} | dirujuk ke LLM: {int(flags.needs_llm.sum())} dari {len(flags)}")

## Skema Tabel 17 terpilih (dipilih di dev; di sini hanya prediksinya)

In [ ]:
h = L.exec_legacy('nb05b_helpers', L.base_namespace())
probs_all = {m: df_t[['pair_id']].merge(test_preds[m][['pair_id'] + C.PROB_COLS], on='pair_id', how='left')[C.PROB_COLS]
             .fillna(0.25).values.astype(np.float32) for m in C.MODELS}
t17 = frozen['table17']
pred17 = L.scheme_predict(t17['selected_key'], h, probs_all, t17['model_weights_dev'], df_t['aspek'], t17['best_T'])
pd.DataFrame({'pair_id': df_t.pair_id, 'skema': t17['selected_key'], 'pred_label': pred17}).to_csv(
    OUT / 'table17_selected_test.csv', index=False, encoding='utf-8-sig')
print('Skema Tabel 17:', t17['selected_key'])

In [ ]:
done = {'finished_at': L.now_utc(), 'dry_run': DRY_RUN, 'n_pairs': len(df_t), 'n_flagged': int(flags.needs_llm.sum()),
        'outputs_sha256': {p.relative_to(OUT).as_posix(): L.sha256_file(p) for p in sorted(OUT.rglob('*.csv'))}}
L.write_json(OUT / 'DONE.json', done)
print(f"✓ Selesai. {len(done['outputs_sha256'])} file prediksi disimpan; tidak ada metrik yang dihitung.")